In [1]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle

In [2]:
# !kaggle datasets download -d shivasingh4945/student-social-media-and-mental-health-impact

In [3]:
# import zipfile
# zip_ref = zipfile.ZipFile('/content/student-social-media-and-mental-health-impact.zip', 'r')
# zip_ref.extractall('/content')
# zip_ref.close()

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

In [ ]:
df = pd.read_csv('/content/Student Social Media And Mental Health Impact.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

**EDA**

In [ ]:
sns.histplot(df['Mental_Health_Score'], kde=True)
plt.show()

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)
plt.show()

In [ ]:
df['Stress_Level'].unique()

In [ ]:
order = ['Low', 'Medium', 'High','Very High']
sns.countplot(data=df, x='Stress_Level', order=order)
plt.show()

In [ ]:
order = ['Low', 'Medium', 'High','Very High']
sns.boxplot(data=df, x='Stress_Level', y='Mental_Health_Score', order=order)
plt.show()

In [ ]:
df.columns

In [ ]:
sns.scatterplot(data=df, x='Avg_Daily_Usage_Hours', y='Mental_Health_Score')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Sleep_Hours_Per_Night', y='Mental_Health_Score')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Physical_Activity_Hours', y='Mental_Health_Score')
plt.show()

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
df['Most_Used_Platform'].value_counts().plot(kind='bar')
plt.show()

**Cheaking Outliers**

In [ ]:
num_feachers = df.select_dtypes(include='number')
Q1 = num_feachers.quantile(0.25)
Q3 = num_feachers.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = ((num_feachers < lower_bound) | (num_feachers > upper_bound))
print(outliers.sum())

**Data Cleaning**

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

In [ ]:
df.describe()

**Skewness**

In [ ]:
num_feachers.skew()

# near to 0 -> Centralized (0.0847)
# -ve -> Left Skewed (-2.56)
# +ve > 0 -> Right Skewed (1.345)

In [ ]:
top_countries = df['Country'].value_counts().index[:10].tolist()

In [ ]:
top_countries

In [ ]:
def group_countries(country):
  if country in top_countries:
    return country
  else:
    return 'Other'

In [ ]:
df['Grouped_Country'] = df['Country'].apply(group_countries)

In [ ]:
df['Grouped_Country'].value_counts()

In [ ]:
df.columns

In [ ]:
from sklearn.model_selection import train_test_split

skewew_col = ['Study_Hours']
other_numric_col = ['Age','Avg_Daily_Usage_Hours','Daily_Unlocks', 'Sleep_Hours_Per_Night', 'Physical_Activity_Hours']

ordinal_col = ['Stress_Level']
normal_col = ['Gender', 'Academic_Level', 'Grouped_Country', 'Purpose_Of_Use', 'Most_Used_Platform']

feature_col = skewew_col + other_numric_col + ordinal_col + normal_col
X = df[feature_col]
y = df['Mental_Health_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
# Add the missing categorical encoders
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

#1. Skewed features
skew_pipeline = Pipeline(steps=[
    ('log_transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

#2. Numeric Features
plain_numeric_pipeline = Pipeline(steps=[
    ('scale', StandardScaler())
])

#3. Ordinal Features (Preserves order, e.g., Low, Medium, High)
ordinal_pipeline = Pipeline(steps=[
    ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High','Very High']]))
])

#4. Nominal Features (No intrinsic order, e.g., Colors, Cities)
nominal_pipeline = Pipeline(steps=[
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('skew_pipeline', skew_pipeline, skewew_col),
    ('plain_numeric_pipeline', plain_numeric_pipeline, other_numric_col),
    ('ordinal_pipeline', ordinal_pipeline, ordinal_col),
    ('nominal_pipeline', nominal_pipeline, normal_col)
])


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error

# Linear Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_pred_train = lr_pipeline.predict(X_train)

lr_r2_testing = r2_score(y_test, lr_pred)
lr_r2_training = r2_score(y_train, lr_pred_train)

lr_mse_testing = mean_squared_error(y_test, lr_pred)
lr_mse_training = mean_squared_error(y_train, lr_pred_train)

lr_mae_testing = mean_absolute_error(y_test, lr_pred)
lr_mae_training = mean_absolute_error(y_train, lr_pred_train)

print("Training R2 Score:", lr_r2_training)
print("Testing R2 Score:", lr_r2_testing)
print("Training MSE:", lr_mse_training)
print("Testing MSE:", lr_mse_testing)
print("Training MAE:", lr_mae_training)
print("Testing MAE:", lr_mae_testing)



# print("Mean Squared Error:", mean_squared_error(y_test, lr_pred))
# print("R2 Score:", r2_score(y_test, lr_pred))
# print("Mean Absolute Error:", mean_absolute_error(y_test, lr_pred))

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error

# Linear Regression
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor())
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_pred_train = rf_pipeline.predict(X_train)

rf_r2_testing = r2_score(y_test, rf_pred)
rf_r2_training = r2_score(y_train, rf_pred_train)

rf_mse_testing = mean_squared_error(y_test, rf_pred)
rf_mse_training = mean_squared_error(y_train, rf_pred_train)

rf_mae_testing = mean_absolute_error(y_test, rf_pred)
rf_mae_training = mean_absolute_error(y_train, rf_pred_train)

print("Training R2 Score:", rf_r2_training)
print("Testing R2 Score:", rf_r2_testing)
print("Training MSE:", rf_mse_training)
print("Testing MSE:", rf_mse_testing)
print("Training MAE:", rf_mae_training)
print("Testing MAE:", rf_mae_testing)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV


param_grid = {
    "random forest__n_estimators": [100, 200, 300, 400, 500],
    "random forest__max_depth": [None, 10, 20, 30],
    "random forest__min_samples_split": [2, 5, 10, 12, 14],
    "random forest__min_samples_leaf": [1, 2, 4,6 , 8, 10]
}

random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='r2',
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)


In [ ]:
rf_best_pipeline = random_search.best_estimator_
rf_best_pipeline

In [ ]:
rf_best_preds = rf_best_pipeline.predict(X_test)

print(f"R2 Score for forest after hyperparameter tuning {r2_score(y_test,rf_best_preds)}")
print(f"MAE Score for forest after hyperparameter tuning {mean_absolute_error(y_test,rf_best_preds)}")
print(f"MSE Score for forest after hyperparameter tuning {mean_squared_error(y_test,rf_best_preds)}")



In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Note: Yeh variables aapke pehle ke code se aane chahiye ---
# Yahan sirf code run karne ke liye dummy variables banaye gaye hain:
y_test = y_train = np.array([1, 2, 3])
X_test = X_train = np.array([[1], [2], [3]])
lr_preds = rf_preds = np.array([1.1, 1.9, 3.2])
lr_r2_testing = rf_r2_testing = 0.9
lr_r2_training = rf_r2_training = 0.95
lr_mae = rf_mae = 0.15

# Mock random_search object simulation ke liye
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- Note: Yeh variables aapke pehle ke code se aane chahiye ---
# Yahan sirf code run karne ke liye dummy variables banaye gaye hain:
y_test = y_train = np.array([1, 2, 3])
X_test = X_train = np.array([[1], [2], [3]])
lr_preds = rf_preds = np.array([1.1, 1.9, 3.2])
lr_r2_testing = rf_r2_testing = 0.9
lr_r2_training = rf_r2_training = 0.95
lr_mae = rf_mae = 0.15

# Mock random_search object simulation ke liye
class MockSearch:
    class best_estimator_:
        def predict(X): return np.array([1.05, 1.95, 3.1])
random_search = MockSearch

# -----------------------------------------------------------
# 1. Linear Regression ke liye RMSE nikalein
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# 2. Default Random Forest ke liye RMSE nikalein
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# 3. Tuned Random Forest ke liye predictions aur metrics nikalein
rf_tuned_preds = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))

rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training = r2_score(y_train, rf_tuned_training_preds)

rf_tuned_mae = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2 = r2_score(y_test, rf_tuned_preds)

# 4. Results ko ek DataFrame mein consolidate karein
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

# 5. Results ko print karein
print(results)

# -----------------------------------------------------------
# 1. Linear Regression ke liye RMSE nikalein
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# 2. Default Random Forest ke liye RMSE nikalein
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# 3. Tuned Random Forest ke liye predictions aur metrics nikalein
rf_tuned_preds = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))

rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training = r2_score(y_train, rf_tuned_training_preds)

rf_tuned_mae = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2 = r2_score(y_test, rf_tuned_preds)

# 4. Results ko ek DataFrame mein consolidate karein
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

# 5. Results ko print karein
print(results)


In [ ]:
from sklearn.metrics import mean_squared_error

# Calculate RMSE for Linear Regression
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

# Calculate RMSE for default Random Forest
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

# Calculate RMSE for tuned Random Forest
rf_tuned_preds         = random_search.best_estimator_.predict(X_test)
rf_tuned_rmse          = np.sqrt(mean_squared_error(y_test, rf_tuned_preds))
rf_tuned_training_preds = random_search.best_estimator_.predict(X_train)
r2_tuned_training      = r2_score(y_train, rf_tuned_training_preds)
rf_tuned_mae           = mean_absolute_error(y_test, rf_tuned_preds)
rf_tuned_r2            = r2_score(y_test, rf_tuned_preds)


# Create a DataFrame to consolidate results
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (default)', 'Random Forest (tuned)'],
    'R2': [lr_r2_testing, rf_r2_testing, rf_tuned_r2],
    'Training R2': [lr_r2_training, rf_r2_training, r2_tuned_training],
    'MAE': [lr_mae, rf_mae, rf_tuned_mae],
    'RMSE': [lr_rmse, rf_rmse, rf_tuned_rmse]
})

print(results)


In [ ]:
# import joblib
# joblib.dump(rf_pipeline, 'mental_health_rf_pipeline.pkl')
# print("Model saved successfully!")